# Setting up the mesh and coordinates of the Nodes

In [ ]:
import numpy as np
import scipy.linalg as scp
import matplotlib.pylab as plt
import time
from scipy.optimize import fsolve
import pandas as pd
from scipy.interpolate import interp1d
from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import NearestNDInterpolator
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
from scipy.linalg import eigvalsh
from scipy.linalg import lu_factor, lu_solve
from scipy.linalg import cho_factor, cho_solve
from scipy.fft import rfft, rfftfreq

from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 15, 6
pass

In [ ]:
L = 11000.0
Le = 6.25 

nEle = max(1, int(round(L / Le)))
actual_Le = L / nEle
nNodes = nEle + 1

x = np.linspace(0, L, nNodes)
y = np.zeros_like(x)
z = np.zeros_like(x)


elements = np.column_stack([np.arange(nEle), np.arange(1, nEle + 1)])

print(f"Beam length             : {L:.3f} m")
print(f"Number of elements      : {nEle}")
print(f"Number of nodes         : {nNodes}")
print(f"Actual element length   : {actual_Le:.3f} m")
print(elements)

fig = plt.figure()
ax = plt.axes(projection="3d")
ax.plot(x, y, z, "-o", lw=2)

ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_zlabel("z (m)")
plt.grid(True)
plt.show()

## Define beam properties

In [ ]:
E = 39e9                                  # Pa, Young's modulus of reinforced concrete
G = 16.3e9                                 # Pa, shear modulus of reinforced concrete
h_out = 6                                  # m, outer diameter of the connector
h_in = 5                                  # m, inner diameter of the connector
w_out = 5.5                                # m, outer width of the connector
w_in = 4.5                                # m, inner width of the connector
t= 0.5
connector_spacing = 250                               # m, spacing between connectors along the beam
hm = 6-t
bm = 5.5-t
Am = hm*bm
pm = 2*(hm+bm)                                                  # m, perimeter of the connector cross-section
Concrete_density = 2600                                         # kg/m3, density of reinforced concrete
TunArea = 6*5.5 - 5*4.5                                         # m2, cross-sectional area of the connector
connector_GJ = G*(2*t**2 * bm**2 * hm**2/(bm+hm))                                 # [N.m2]
connector_Im = 2600* (bm**2 + hm**2)**2 * bm*hm*t / 144         #[kg.m] (Volgens mij hoort dit niet zo)
connector_m =  (TunArea) *2600                                  # [kg/m]
connector_EIy = 1/12 * (w_out**3*h_out - w_in**3*h_in) * E      # [N.m2]
connector_EIz = 1/12 * (w_out*h_out**3 - w_in*h_in**3) * E      # [N.m2]
connector_EA = (6 * 5.5 - 5 * 4.5) * E     # [N]
GJ_eff_connector = connector_GJ / connector_spacing
Im_eff_connector = connector_Im / connector_spacing
EIy_eff_connector = connector_EIy / connector_spacing
EIz_eff_connector = connector_EIz / connector_spacing
EA_eff_connector = connector_EA / connector_spacing
m_eff_connector = connector_m / connector_spacing
Beam_GJ = 9.5e12 * 2 +  GJ_eff_connector             # [N.m2]
Beam_Im = 1.5e6 * 2 +  Im_eff_connector                     #[kg.m]
Beam_m = 1.1e5 * 2 +  m_eff_connector                 # [kg/m]
Beam_EIy = 141193e9 * 2 +  EIy_eff_connector              # [N.m2]
Beam_EIz = 141193e9 * 2 +  EIz_eff_connector               # [N.m2]
Beam_EA = 15.5*39e9 * 2 +  EA_eff_connector              # [N]

# Mesh setup


In [ ]:
NodeC = [[x, y, z] for x, y, z in zip(x, y, z)]
nNodes = len(NodeC)
Ele_tunnel = [[n1, n1 + 1, Beam_m, Beam_EA, Beam_EIy, Beam_EIz, Beam_GJ, Beam_Im] for n1 in range(0, nNodes - 1)]
escape_spacing = 250.0
NodeArr = np.array(NodeC, dtype=float)

print(f"Total nodes : {nNodes}")
print(NodeC)
print(NodeArr)
print(len(NodeArr))

# Add mooring 

In [ ]:
mooring_spacing = 25.0

s = np.linspace(0, L, nNodes)
target_spacing = np.arange(0.0, L + 0.5 * mooring_spacing, mooring_spacing)

mooring_nodes = np.unique([int(np.argmin(np.abs(s - target))) for target in target_spacing])

print("Mooring target positions:", target_spacing)
print("Mooring node indices:", mooring_nodes)
print("Mooring node positions:", s[mooring_nodes])


# Global Matrixes

In [ ]:
LDOF = 6
nDof = LDOF*nNodes 
K = np.zeros(nDof*nDof)
M = np.zeros(nDof*nDof)
Q = np.zeros(nDof*nDof)

from BeamMatrices import Beam3DMatrices

for iEle in range(0, nEle):
    n1, n2, m, EA, EIy, EIz, GJ, Im = Ele_tunnel[iEle]
    n1 = int(round(n1))
    n2 = int(round(n2))
    
    n1dof = LDOF*n1 + np.arange(0,LDOF)
    n2dof = LDOF*n2 + np.arange(0,LDOF)    
    
    Me, Ke, Qe = Beam3DMatrices(m, EA, EIy, EIz, GJ, Im, (NodeC[n1], NodeC[n2]) )
    
    indexes = np.append(n1dof, n2dof)
    for i in range(0, 2*LDOF):
        for j in range(0, 2*LDOF):
            ij = indexes[i]*nDof + indexes[j]
            #print(ij)
            M[ij] = M[ij] + Me[i,j]
            K[ij] = K[ij] + Ke[i,j]
            Q[ij] = Q[ij] + Qe[i,j]

M = M.reshape((nDof, nDof))
K = K.reshape((nDof, nDof))
Q = Q.reshape((nDof, nDof))

M_lumped = np.zeros_like(M)

for iEle in range(nEle):
    n1, n2, m, EA, EIy, EIz, GJ, Im = Ele_tunnel[iEle]
    n1 = int(round(n1))
    n2 = int(round(n2))

    L_ele = np.linalg.norm(np.array(NodeC[n2]) - np.array(NodeC[n1]))

    M_node = m * L_ele / 2
    Jx_node = Beam_Im * L_ele / 2
    J_bend_y = Beam_EIy * L_ele / 2
    J_bend_z = Beam_EIz * L_ele / 2

    for node in [n1, n2]:
        i = node * LDOF

        M_lumped[i + 0, i + 0] += M_node
        M_lumped[i + 1, i + 1] += M_node
        M_lumped[i + 2, i + 2] += M_node
        M_lumped[i + 3, i + 3] += Jx_node
        M_lumped[i + 4, i + 4] += J_bend_y
        M_lumped[i + 5, i + 5] += J_bend_z

M = M_lumped


# Add Added-Mass, Mooring stiffness and Added Damping

In [ ]:
rho_w = 1027.0     
Cm = 1.0           
D_outer = 12.7      

A_disp = np.pi * D_outer**2 / 4

m_added = rho_w * Cm * A_disp
M_added = m_added * Q
M = M + M_added

print("Added mass per meter:", round(m_added, 2), "kg/m")
print("Structural mass per meter:", round(Beam_m, 2), "kg/m")
print("Added/structural mass ratio:", round(m_added / Beam_m, 2))


n_stiff = 360
half_n = n_stiff // 2

k_start_v = 45
k_start_h = 161

k_mid_v = 60
k_mid_h = 396

k_end_v = 81
k_end_h = 843

k_v = np.linspace(k_start_v, k_mid_v, half_n + 1)
k_h = np.linspace(k_start_h, k_mid_h, half_n + 1)


print(k_v)
print(k_h)

In [ ]:
plt.figure()
plt.spy(M)
plt.title("Mass matrix")
plt.figure()
plt.spy(K)
plt.title("Stiffness matrix")
K_sum_before = np.sum(K)
print("Sum of stiffness matrix before applying boundary conditions:", K_sum_before)
pass

# Matrixes after adding stiffness and Mass

In [ ]:
K_before = K.copy()

cat_i = 0
added = 0.0

def AddMooringSpringHV(K, node_id, kh, kv, LDOF=6):
    i = int(node_id) * LDOF
    K[i + 1, i + 1] += kh   # y-direction, horizontal transverse
    K[i + 2, i + 2] += kv   # z-direction, vertical
    return K

for node in mooring_nodes:
    s_moor = s[node]

    if s_moor < 1000:
        kh = 120e3
        kv = 85e3

    elif 1000 <= s_moor < 10000:
        idx = min(cat_i, len(k_h) - 1)
        kh = k_h[idx] * 1e2
        kv = k_v[idx] * 1e3
        cat_i += 1

    else:
        kh = 200e3
        kv = 85e3

    K = AddMooringSpringHV(K, node, kh, kv, LDOF=LDOF)
    added += kh + kv

K_after = K.copy()

print("Expected added stiffness:", added)
print("Actual added diagonal stiffness:", np.sum(np.diag(K_after - K_before)))

for node in mooring_nodes:
    dof_y = node * LDOF + 1
    dof_z = node * LDOF + 2

    print(f"Node {node}: " f"ΔK_uy = {K_after[dof_y, dof_y] - K_before[dof_y, dof_y]:.3e}, " f"ΔK_uz = {K_after[dof_z, dof_z] - K_before[dof_z, dof_z]:.3e}")

for i in range(min(10, nNodes)):
    print( M[i*LDOF+3, i*LDOF+3], M[i*LDOF+4, i*LDOF+4],  M[i*LDOF+5, i*LDOF+5])

In [ ]:
NodesClamp = (0, nNodes-1)
DofsP = np.empty([0], dtype=int)
for n0 in NodesClamp:
    DofsP = np.append(DofsP, n0*LDOF + np.arange(0,LDOF))

DofsP = np.empty([0], dtype=int)
for n0 in NodesClamp: DofsP = np.append(DofsP, n0*LDOF + np.array([0,1,2]))

DofsF = np.arange(0, nDof)
DofsF = np.delete(DofsF, DofsP) 

print(DofsP)
print(DofsF)

def AddRotationalSpring(K, node_id, krx, kry, krz, LDOF=6):

    i = node_id * LDOF

    K[i+3, i+3] += krx
    K[i+4, i+4] += kry
    K[i+5, i+5] += krz

    return K


# Assumption made for 100 and 10 times stiffness at BC!

krx_end = Beam_Im * 100  
kry_end = Beam_EIy * 10  
krz_end = Beam_EIz * 10 

for node in NodesClamp:
    K = AddRotationalSpring(K, node, krx_end, kry_end, krz_end, LDOF=LDOF)

M_FF = M[np.ix_(DofsF, DofsF)]
K_FF = K[np.ix_(DofsF, DofsF)]
Q_FF = Q[np.ix_(DofsF, DofsF)]

print("K shape:", K.shape)
print("K_FF shape:", K_FF.shape)
print("Number of prescribed DOFs:", len(DofsP))
print("Number of free DOFs:", len(DofsF))
pass

# Eigen frequencies and Modal Shapes

In [ ]:
m_eigs = eigvalsh(M_FF)
print("Smallest eigenvalues of M_FF:")
print(m_eigs[:20])
print("Number <= 0:", np.sum(m_eigs <= 0))

bad_modes = np.where(m_eigs <= 0)[0]

In [ ]:
Me, _, _ = Beam3DMatrices(Beam_m, Beam_EA, Beam_EIy, Beam_EIz, Beam_GJ, Beam_Im, (NodeC[0], NodeC[1]))
eig = np.linalg.eigvalsh(Me)
print(np.min(eig))
print(eig)

In [ ]:
n_modes = 500
w2, vr = eigh(K_FF, M_FF)
w2 = w2[:n_modes]
vr = vr[:, :n_modes]
w = np.sqrt(np.maximum(w2, 0))
f = w / (2*np.pi)

print(f)

In [ ]:
Phi = np.zeros((nDof, n_modes))

for i in range(n_modes):
    Phi[DofsF, i] = vr[:, i]


def plot_mode(mode_id, scale=1.0, direction="z"):
    phi = Phi[:, mode_id]
    ux = phi[0::LDOF]
    uy = phi[1::LDOF]
    uz = phi[2::LDOF]

    if direction == "y":
        deformation = uy
        ylabel = "y displacement mode shape"
    elif direction == "z":
        deformation = uz
        ylabel = "z displacement mode shape"
    else:
        raise ValueError("direction must be 'y' or 'z'")

    deformation = deformation / np.max(np.abs(deformation))

    plt.figure()
    plt.plot(s, deformation, "-o")
    plt.xlabel("Beam coordinate s [m]")
    plt.ylabel(ylabel)
    plt.title(f"Mode {mode_id + 1}, f = {f[mode_id]:.4f} Hz")
    plt.grid(True)
    plt.show()



def plot_mode_3d(mode_id, scale=5.0):
    phi = Phi[:, mode_id]

    ux = phi[0::LDOF]
    uy = phi[1::LDOF]
    uz = phi[2::LDOF]

    max_disp = max(
        np.max(np.abs(ux)),
        np.max(np.abs(uy)),
        np.max(np.abs(uz))
    )

    ux = ux / max_disp
    uy = uy / max_disp
    uz = uz / max_disp

    x_def = x + scale * ux
    y_def = y + scale * uy
    z_def = z + scale * uz

    fig = plt.figure()
    ax = plt.axes(projection="3d")

    ax.plot(x, y, z, "k--", label="undeformed")
    ax.plot(x_def, y_def, z_def, "-o", label="mode shape")

    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.set_title(f"Mode {mode_id + 1}, f = {f[mode_id]:.4f} Hz")
    ax.legend()
    plt.show()


In [ ]:
plt.figure()
plt.plot(np.arange(1, len(f) + 1), f, "o-")
plt.xlabel("Mode number")
plt.ylabel("Frequency [Hz]")
plt.title("Natural frequencies")
plt.grid(True)
plt.show()
print(f[:10])

# Plotting Modal Shapes with corresponding Freq

In [ ]:
n_modes_available = vr.shape[1]
Phi = np.zeros((nDof, n_modes_available))
Phi[DofsF, :] = vr

def plot_mode_shapes_subplots(Phi, f, s, nModesPlot=12, direction="z"):

    dof_map = {"x": 0, "y": 1, "z": 2, "rx": 3, "ry": 4, "rz": 5,}
    local_dof = dof_map[direction]
    nModesPlot = min(nModesPlot, Phi.shape[1])
    ncols = 3
    nrows = int(np.ceil(nModesPlot / ncols))

    fig, axs = plt.subplots(nrows, ncols, figsize=(14, 3.5*nrows), sharex=True)
    axs = np.asarray(axs).ravel()

    for mode in range(nModesPlot):
        shape = Phi[local_dof::LDOF, mode]
        max_val = np.max(np.abs(shape))
        if max_val > 0:
            shape = shape / max_val

        axs[mode].plot(s, shape, "-o", markersize=3)
        axs[mode].axhline(0, linewidth=0.8)
        axs[mode].grid(True)
        axs[mode].set_title(f"Mode {mode+1}: f = {f[mode]:.4f} Hz")
        axs[mode].set_ylabel(direction)

    for ax in axs[nModesPlot:]:
        ax.axis("off")

    for ax in axs[-ncols:]:
        ax.set_xlabel("s [m]")

    fig.suptitle(f"Modal shapes in {direction}-direction", fontsize=14)
    plt.tight_layout()
    plt.show()

### Vertical Modal shapes

In [ ]:
plot_mode_shapes_subplots(Phi, f, s, nModesPlot=12, direction="z")

### Horizontal Modal shapes

In [ ]:
plot_mode_shapes_subplots(Phi, f, s, nModesPlot=12, direction="y")

### Rotational Modal shapes

In [ ]:
plot_mode_shapes_subplots(Phi, f, s, nModesPlot=12, direction="rx")

## Solving the Model Staticly

In [ ]:
qz = 250e3
mooring_restoring_force = 2900e3
element_length_factor = 25 / Le

Fz = np.zeros(nDof)

for iEle in range(nEle):
    n1 = int(Ele_tunnel[iEle][0])
    n2 = int(Ele_tunnel[iEle][1])

    Le = np.linalg.norm(np.asarray(NodeC[n2]) - np.asarray(NodeC[n1]))

    Fz[n1*LDOF + 2] += (qz * Le / 2) - (mooring_restoring_force / element_length_factor)
    Fz[n2*LDOF + 2] += (qz * Le / 2) - (mooring_restoring_force / element_length_factor)

F_Fz = Fz[DofsF]
u_Fz = np.linalg.solve(K_FF, F_Fz)
uz = np.zeros(nDof)
uz[DofsF] = u_Fz    

qy = 11e3 

Fy = np.zeros(nDof)

for iEle in range(nEle):
    n1 = int(Ele_tunnel[iEle][0])
    n2 = int(Ele_tunnel[iEle][1])

    Le = np.linalg.norm(np.asarray(NodeC[n2]) - np.asarray(NodeC[n1]))

    Fy[n1*LDOF + 1] += qy * Le / 2
    Fy[n2*LDOF + 1] += qy * Le / 2

F_Fy = Fy[DofsF]
print(F_Fy[:30])
print(np.max(np.abs(F_Fy)))

u_Fy = np.linalg.solve(K_FF, F_Fy)

uy = np.zeros(nDof)
uy[DofsF] = u_Fy


In [ ]:
uz = uz[2::LDOF]

plt.figure()
plt.plot(s, uz, "-o")
plt.xlabel("s [m]")
plt.ylabel("Vertical displacement uz [m]")
plt.title("Static vertical displacement")
plt.grid()
plt.show()

## Horizontal Displacement

In [ ]:
uy = uy[1::LDOF]

plt.figure()
plt.plot(s, uy, "-o")
plt.xlabel("s [m]")
plt.ylabel("Horizontal displacement uy [m]")
plt.title("Static horizontal displacement")
plt.grid()
plt.show()


# Dynamic modelling

## Dynamic Forcing

In [ ]:
F_static = (Fy + Fz).copy()
dt = 0.2
T_total = 500
t = np.arange(0, T_total + dt, dt)
n_steps = len(t)

Tp = 6.75
h = 44

H_wave = 2.4

n_dof = nDof

dfy = pd.read_csv("interpolated_horizontal_stiffness_data.csv")
dfz = pd.read_csv("interpolated_vertical_stiffness_data.csv")

ky_lin = LinearNDInterpolator(list(zip(dfy["water_depth_m"], dfy["u_mid_m"])), dfy["Ky_N_per_m"])
ky_near = NearestNDInterpolator(list(zip(dfy["water_depth_m"], dfy["u_mid_m"])), dfy["Ky_N_per_m"])

kz_lin = LinearNDInterpolator(list(zip(dfz["water_depth_m"], dfz["u_mid_m"])), dfz["Kz_N_per_m"])
kz_near = NearestNDInterpolator(list(zip(dfz["water_depth_m"], dfz["u_mid_m"])), dfz["Kz_N_per_m"])

def safe_interp(lin, near, h, u):
    val = lin(h, u)
    if np.isnan(val):
        val = near(h, u)
    return float(val)

depth_nodes = np.linspace(160, 275, len(mooring_nodes))

def airy_wave_properties(T, h, g=9.81):
    omega = 2*np.pi/T

    def residual(k):
        return g*k*np.tanh(k*h) - omega**2

    k0 = omega**2/g
    k = fsolve(residual, k0)[0]
    L = 2*np.pi/k
    c = omega/k

    return omega, k, L, c


omega, k, L, c = airy_wave_properties(Tp, h)


def airy_horizontal_velocity(z, x, t, H, T, h, g=9.81):
    a = H / 2
    omega, k, _, _ = airy_wave_properties(T, h, g)
    return (a * omega * np.cosh(k * (z + h)) / np.sinh(k * h) * np.cos(k*x - omega*t))


def airy_horizontal_acceleration(z, x, t, H, T, h, g=9.81):
    a = H / 2
    omega, k, _, _ = airy_wave_properties(T, h, g)
    return (a * omega**2 * np.cosh(k * (z + h)) / np.sinh(k * h) * np.sin(k*x - omega*t))


def morison_force_yz_per_length(z, x, t, H, T, h, D, Cd_y, Cm_y, Cd_z, Cm_z, rho=1027, g=9.81):

    # y
    uy = airy_horizontal_velocity(z, x, t, H, T, h, g)
    ay = airy_horizontal_acceleration(z, x, t, H, T, h, g)

    # z
    a = H / 2
    omega, k, _, _ = airy_wave_properties(T, h, g)

    uz = (a * omega * np.sinh(k * (z + h)) / np.sinh(k * h) * np.sin(k*x - omega*t))

    az = (-a * omega**2 * np.sinh(k * (z + h)) / np.sinh(k * h) * np.cos(k*x - omega*t))

    current_y = 0.8
    u_rel_y = uy + current_y

    qy = (0.5 * rho * Cd_y * D * abs(u_rel_y) * u_rel_y + rho * Cm_y * (np.pi * D**2 / 4) * ay)

    qz = (0.5 * rho * Cd_z * D * abs(uz) * uz + rho * Cm_z * (np.pi * D**2 / 4) * az)

    return qy, qz

F_time = np.zeros((n_steps, nDof))

D = 12.7
Cd_y = 0.7
Cm_y = Cm
Cd_z = 0.7
Cm_z = Cm

for n, tn in enumerate(t):

    Fdyn = np.zeros(nDof)

    for iEle in range(nEle):

        n1 = int(Ele_tunnel[iEle][0])
        n2 = int(Ele_tunnel[iEle][1])

        x1, z1 = NodeC[n1][0], NodeC[n1][2]
        x2, z2 = NodeC[n2][0], NodeC[n2][2]

        Le = np.linalg.norm(np.asarray(NodeC[n2]) - np.asarray(NodeC[n1]))

        qy1, qz1 = morison_force_yz_per_length(z1, x1, tn, H_wave, Tp, h, D, Cd_y, Cm_y, Cd_z, Cm_z)

        qy2, qz2 = morison_force_yz_per_length(z2, x2, tn, H_wave, Tp, h, D, Cd_y, Cm_y, Cd_z, Cm_z)

        # y
        Fdyn[n1*LDOF + 1] += Le / 6 * (2*qy1 + qy2)
        Fdyn[n2*LDOF + 1] += Le / 6 * (qy1 + 2*qy2)

        # z
        Fdyn[n1*LDOF + 2] += Le / 6 * (2*qz1 + qz2)
        Fdyn[n2*LDOF + 2] += Le / 6 * (qz1 + 2*qz2)

    F_time[n, :] = Fdyn


alpha_R = 0.01
beta_R  = 1e-4

C_FF = alpha_R * M_FF + beta_R * K_FF

beta = 1/4
gamma = 1/2

n_free = len(DofsF)

U_hist = np.zeros((n_steps, n_free))
V_hist = np.zeros((n_steps, n_free))
A_hist = np.zeros((n_steps, n_free))

u = np.zeros(n_free)
v = np.zeros(n_free)

F0 = F_static + F_time[0, :]
F0_F = F0[DofsF]

a = np.linalg.solve(M_FF, F0_F - C_FF @ v - K_FF @ u)

K_eff = (K_FF  + gamma / (beta * dt) * C_FF + 1 / (beta * dt**2) * M_FF)

F_static_F = F_static[DofsF]

u = np.linalg.solve(K_FF, F_static_F)
v = np.zeros(len(DofsF))
a = np.zeros(len(DofsF))

u_max_y = dfy["u_mid_m"].max()
u_max_z = dfz["u_mid_m"].max()

free_dof_map = {gdof: i for i, gdof in enumerate(DofsF)}

u = np.linalg.solve(K_FF, F_static_F)

for it in range(20):

    K_static = K_FF.copy()

    for i, node in enumerate(mooring_nodes):

        h_node = depth_nodes[i]

        global_dof_y = node*LDOF + 1
        global_dof_z = node*LDOF + 2

        if global_dof_y in free_dof_map:
            local_dof_y = free_dof_map[global_dof_y]
            uy_abs = np.clip(abs(u[local_dof_y]), 0.0, u_max_y)
            ky = safe_interp(ky_lin, ky_near, h_node, uy_abs)
            K_static[local_dof_y, local_dof_y] += ky

        if global_dof_z in free_dof_map:
            local_dof_z = free_dof_map[global_dof_z]
            uz_abs = np.clip(abs(u[local_dof_z]), 0.0, u_max_z)
            kz = safe_interp(kz_lin, kz_near, h_node, uz_abs)
            K_static[local_dof_z, local_dof_z] += kz

    u_new = np.linalg.solve(K_static, F_static_F)

    if np.linalg.norm(u_new - u) / np.linalg.norm(u_new) < 1e-6:
        print(f"Static mooring equilibrium converged in {it+1} iterations")
        break

    u = u_new

v = np.zeros(len(DofsF))
a = np.zeros(len(DofsF))

for n in range(n_steps):

    if np.any(np.isnan(u)):
        idx = np.where(np.isnan(u))[0][0]
        print(f"\nNaN detected in u at step {n}")
        print(f"DOF = {idx}")
        raise RuntimeError("NaN in displacement vector")

    K_nl = K_FF.copy()

    for i, node in enumerate(mooring_nodes):
        h_node = depth_nodes[i]
        global_dof_y = node*LDOF + 1
        global_dof_z = node*LDOF + 2

        if global_dof_y in free_dof_map:
            local_dof_y = free_dof_map[global_dof_y]
            uy = u[local_dof_y]
            uy_abs = np.clip(abs(uy), 0.0, u_max_y)
            ky_raw = ky_lin(h_node, uy_abs)

            if np.isnan(ky_raw):
                print("\nFIRST NaN IN KY INTERPOLATION")
                print(f"step       = {n}")
                print(f"node       = {node}")
                print(f"h_node     = {h_node}")
                print(f"uy         = {uy}")
                print(f"uy_abs     = {uy_abs}")
                raise RuntimeError("NaN from ky interpolation")

            ky = float(ky_raw)
            if np.isnan(ky):
                raise RuntimeError("ky became NaN")
            K_nl[local_dof_y, local_dof_y] += ky

        if global_dof_z in free_dof_map:
            local_dof_z = free_dof_map[global_dof_z]
            uz = u[local_dof_z]
            uz_abs = np.clip(abs(uz), 0.0, u_max_z)
            kz_raw = kz_lin(h_node, uz_abs)

            if np.isnan(kz_raw):
                print("\nFIRST NaN IN KZ INTERPOLATION")
                print(f"step       = {n}")
                print(f"node       = {node}")
                print(f"h_node     = {h_node}")
                print(f"uz         = {uz}")
                print(f"uz_abs     = {uz_abs}")
                raise RuntimeError("NaN from kz interpolation")

            kz = float(kz_raw)
            if np.isnan(kz):
                raise RuntimeError("kz became NaN")
            K_nl[local_dof_z, local_dof_z] += kz

    K_eff = (K_nl + gamma/(beta*dt)*C_FF + 1/(beta*dt**2)*M_FF)

    F_ext_F = F_static_F + F_time[n, DofsF]
    # continue with F_eff, solve u_new, update a_new/v_new

    F_eff = (F_ext_F  + M_FF @ ( 1 / (beta * dt**2) * u + 1 / (beta * dt) * v + (1 / (2 * beta) - 1) * a) + C_FF @ (gamma / (beta * dt) * u + (gamma / beta - 1) * v + dt * (gamma / (2 * beta) - 1) * a))

    u_new = np.linalg.solve(K_eff, F_eff)
    a_new = (1 / (beta * dt**2) * (u_new - u) - 1 / (beta * dt) * v - (1 / (2 * beta) - 1) * a)
    v_new = v + dt * ((1 - gamma) * a + gamma * a_new)

    U_hist[n, :] = u_new
    V_hist[n, :] = v_new
    A_hist[n, :] = a_new

    u = u_new
    v = v_new
    a = a_new



U_full = np.zeros((n_steps, nDof))
U_full[:, DofsF] = U_hist



### Displacement of the Tunnel per time step

In [ ]:
plot_steps = [0, n_steps // 3, 2 * n_steps // 3, n_steps - 1]

plt.figure(figsize=(10, 6))

for n in plot_steps:
    scale = 1 
    y_def = []
    z_def = []

    for node in range(nNodes):
        y = NodeC[node][1]
        z = NodeC[node][2]

        uy = U_full[n, node*LDOF + 1]
        uz = U_full[n, node*LDOF + 2]

        y_def.append(y + scale * uy)
        z_def.append(z + scale * uz)

    plt.plot(s, z_def, label=f't = {t[n]:.1f} s')

plt.xlabel('x [m]')
plt.ylabel('z [m]')
plt.title('Deformed shape at selected time steps')
plt.grid()
plt.legend()
plt.show()

In [ ]:
plot_steps = np.linspace(0, n_steps - 1, 10, dtype=int)

plt.figure(figsize=(10, 6))

for n in plot_steps:
    scale = 10 
    y_def = []
    z_def = []

    for node in range(nNodes):
        y = NodeC[node][1]
        z = NodeC[node][2]

        uy = U_full[n, node*LDOF + 1]
        uz = U_full[n, node*LDOF + 2]

        y_def.append(y + scale * uy)
        z_def.append(z + scale * uz)

    plt.plot(s, y_def,label=f't = {t[n]:.1f} s')

plt.xlabel('x [m]')
plt.ylabel('y [m]')
plt.title('Deformed shape at selected time steps')
plt.grid()
plt.legend()
plt.show()

## Displacement of A node 

In [ ]:
node = nNodes // 2

uy_hist = U_full[:, node*LDOF + 1]
uz_hist = U_full[:, node*LDOF + 2]

plt.figure(figsize=(10,5))
plt.plot(t, uy_hist, label='u_y')
plt.xlabel('Time [s]')
plt.ylabel('Displacement y [m]')
plt.title(f'Displacement history of node {node}')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(t, uz_hist, label='u_z')
plt.xlabel('Time [s]')
plt.ylabel('Displacement [m]')
plt.title(f'Displacement history of node {node}')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
node_mid = len(NodeC)//2
dof = node_mid*LDOF + 1
dof2 = node_mid*LDOF + 2

plt.plot(t, F_time[:, dof]/1000, label='Horizontal wave force (kN)')
plt.xlabel("Time [s]")
plt.ylabel("Wave force [kN]")
plt.grid()
plt.legend()    
plt.show()

In [ ]:
plt.plot(t, F_time[:, dof2]/1000, label='Vertical wave force (kN)')
plt.xlabel("Time [s]")
plt.ylabel("Wave force [kN]")
plt.grid()
plt.legend()    
plt.show()

In [ ]:
node_mid = len(NodeC)//2
dof_y = node_mid*LDOF + 1

sig = U_full[:, dof_y] - np.mean(U_full[:, dof_y])
freq = rfftfreq(len(sig), dt)
amp = np.abs(rfft(sig))

print("-" * 50)
print(f'The maximum frequency is {freq[-1]:.4f} Hz')
print(f'The frequency resolution is {freq[1]-freq[0]:.4f} Hz')
print(f'The dominant frequency is {freq[np.argmax(amp)]:.4f} Hz')
print("-" * 50)
print(f'The maximum amplitude is {np.max(amp):.4e} m')

plt.plot(freq, amp)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("FFT of horizontal displacement at mid-node")
plt.xlim(0, 0.5)
plt.grid()
plt.show()

## Reduced-order model

In [ ]:
# ROM settings
rom_mode_counts = [10, 20, 50, 100, min(500, vr.shape[1])]
rom_mode_counts = sorted(set([m for m in rom_mode_counts if m <= vr.shape[1]]))

mid_node = int(0.50 * (nNodes - 1))
mid_dof_y_global = mid_node * LDOF + 1
mid_dof_z_global = mid_node * LDOF + 2

free_dof_map = {gdof: i for i, gdof in enumerate(DofsF)}
mid_dof_y = free_dof_map[mid_dof_y_global]
mid_dof_z = free_dof_map[mid_dof_z_global]

uy_full = U_full[:, mid_dof_y_global]
uz_full = U_full[:, mid_dof_z_global]

alpha_R = 0.01
beta_R = 1e-4

print("ROM mode counts:", rom_mode_counts)

In [ ]:
rom_results = {}
rom_summary = []

for n_modes_rom in rom_mode_counts:

    start_time = time.time()

    Phi_r = vr[:, :n_modes_rom]

    M_r = Phi_r.T @ M_FF @ Phi_r
    K_r = Phi_r.T @ K_FF @ Phi_r
    C_r = alpha_R * M_r + beta_R * K_r

    q_static = np.linalg.solve(K_r, Phi_r.T @ F_static_F)

    q = np.zeros(n_modes_rom)
    qdot = np.zeros(n_modes_rom)
    qddot = np.linalg.solve(M_r, Phi_r.T @ F_time[0, DofsF] - C_r @ qdot - K_r @ q)

    K_eff_r = K_r + gamma/(beta*dt) * C_r + 1/(beta*dt**2) * M_r

    q_hist = np.zeros((n_steps, n_modes_rom))
    uy_rom = np.zeros(n_steps)
    uz_rom = np.zeros(n_steps)

    for i in range(n_steps):

        F_r = Phi_r.T @ F_time[i, DofsF]

        F_eff_r = (
            F_r
            + M_r @ (
                1/(beta*dt**2)*q
                + 1/(beta*dt)*qdot
                + (1/(2*beta) - 1)*qddot
            )
            + C_r @ (
                gamma/(beta*dt)*q
                + (gamma/beta - 1)*qdot
                + dt*(gamma/(2*beta) - 1)*qddot
            )
        )

        q_new = np.linalg.solve(K_eff_r, F_eff_r)

        qddot_new = (
            1/(beta*dt**2)*(q_new - q)
            - 1/(beta*dt)*qdot
            - (1/(2*beta) - 1)*qddot
        )

        qdot_new = qdot + dt*((1 - gamma)*qddot + gamma*qddot_new)

        u_mid_r = Phi_r[[mid_dof_y, mid_dof_z], :] @ (q_static + q_new)
        uy_rom[i] = u_mid_r[0]
        uz_rom[i] = u_mid_r[1]

        q_hist[i, :] = q_new
        q = q_new
        qdot = qdot_new
        qddot = qddot_new

    runtime = time.time() - start_time

    err_uy = np.max(np.abs(uy_rom - uy_full)) / max(np.max(np.abs(uy_full)), 1e-30) * 100
    err_uz = np.max(np.abs(uz_rom - uz_full)) / max(np.max(np.abs(uz_full)), 1e-30) * 100

    rom_results[n_modes_rom] = {
        "uy": uy_rom,
        "uz": uz_rom,
        "q_hist": q_hist,
    }

    rom_summary.append({
        "Used modes": n_modes_rom,
        "Max error uy [%]": err_uy,
        "Max error uz [%]": err_uz,
        "Max uy ROM [m]": np.max(np.abs(uy_rom)),
        "Max uz ROM [m]": np.max(np.abs(uz_rom)),
        "Runtime [min]": runtime / 60,
    })

df_rom = pd.DataFrame(rom_summary)
display(df_rom)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(t, uy_full, "k", label="Full model")

for n_modes_rom in rom_mode_counts:
    plt.plot(t, rom_results[n_modes_rom]["uy"], label=f"ROM {n_modes_rom} modes")

plt.xlabel("Time [s]")
plt.ylabel(r"$u_y$ [m]")
plt.title("ROM convergence for horizontal midspan displacement")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(t, uz_full, "k", label="Full model")

for n_modes_rom in rom_mode_counts:
    plt.plot(t, rom_results[n_modes_rom]["uz"], label=f"ROM {n_modes_rom} modes")

plt.xlabel("Time [s]")
plt.ylabel(r"$u_z$ [m]")
plt.title("ROM convergence for vertical midspan displacement")
plt.grid(True)
plt.legend()
plt.show()

## Dynamic response at key locations

In [ ]:
key_nodes = {
    "quarter_span": int(0.25 * (nNodes - 1)),
    "mid_span": int(0.50 * (nNodes - 1)),
    "three_quarter_span": int(0.75 * (nNodes - 1)),
}

uy_all = U_full[:, 1::LDOF]
uz_all = U_full[:, 2::LDOF]

key_nodes["max_uy_node"] = np.unravel_index(np.argmax(np.abs(uy_all)), uy_all.shape)[1]
key_nodes["max_uz_node"] = np.unravel_index(np.argmax(np.abs(uz_all)), uz_all.shape)[1]
key_nodes["middle_mooring_node"] = mooring_nodes[len(mooring_nodes)//2]

response_summary = []

for name, node in key_nodes.items():

    dof_y = node * LDOF + 1
    dof_z = node * LDOF + 2

    uy = U_full[:, dof_y]
    uz = U_full[:, dof_z]

    response_summary.append({
        "location": name,
        "node": node,
        "x [m]": NodeC[node][0],
        "max uy [m]": np.max(np.abs(uy)),
        "max uz [m]": np.max(np.abs(uz)),
        "rms uy [m]": np.sqrt(np.mean(uy**2)),
        "rms uz [m]": np.sqrt(np.mean(uz**2)),
    })

df_response = pd.DataFrame(response_summary)
display(df_response)

In [ ]:
plt.figure(figsize=(10, 5))

for name in ["quarter_span", "mid_span", "three_quarter_span"]:
    node = key_nodes[name]
    dof_y = node * LDOF + 1
    plt.plot(t, U_full[:, dof_y], label=name.replace("_", " "))

plt.xlabel("Time [s]")
plt.ylabel(r"$u_y$ [m]")
plt.title("Horizontal displacement histories")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))

for name in ["quarter_span", "mid_span", "three_quarter_span"]:
    node = key_nodes[name]
    dof_z = node * LDOF + 2
    plt.plot(t, U_full[:, dof_z], label=name.replace("_", " "))

plt.xlabel("Time [s]")
plt.ylabel(r"$u_z$ [m]")
plt.title("Vertical displacement histories")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
for name, node in key_nodes.items():

    dof_y = node * LDOF + 1
    dof_z = node * LDOF + 2

    for direction, dof in [("y", dof_y), ("z", dof_z)]:
        signal = U_full[:, dof] - np.mean(U_full[:, dof])
        freq = rfftfreq(len(signal), dt)
        amp = 2*np.abs(rfft(signal)) / len(signal)

        plt.figure(figsize=(10, 4))
        plt.plot(freq, amp)
        plt.xlim(0, 0.5)
        plt.xlabel("Frequency [Hz]")
        plt.ylabel("Amplitude [m]")
        plt.title(f"Frequency spectrum of u_{direction} at {name}")
        plt.grid(True)
        plt.show()

## Mooring force envelopes

In [ ]:
n_moor = len(mooring_nodes)

Fy_moor = np.zeros((len(t), n_moor))
Fz_moor = np.zeros((len(t), n_moor))
Fres_moor = np.zeros((len(t), n_moor))

for j, node in enumerate(mooring_nodes):

    h_node = depth_nodes[j]
    dof_y = node * LDOF + 1
    dof_z = node * LDOF + 2

    uy = U_full[:, dof_y]
    uz = U_full[:, dof_z]

    for i in range(len(t)):

        uy_abs = np.clip(abs(uy[i]), 0.0, u_max_y)
        uz_abs = np.clip(abs(uz[i]), 0.0, u_max_z)

        ky = safe_interp(ky_lin, ky_near, h_node, uy_abs)
        kz = safe_interp(kz_lin, kz_near, h_node, uz_abs)

        Fy_moor[i, j] = ky * uy[i]
        Fz_moor[i, j] = kz * uz[i]
        Fres_moor[i, j] = np.sqrt(Fy_moor[i, j]**2 + Fz_moor[i, j]**2)

max_per_mooring = np.max(np.abs(Fres_moor), axis=0)
governing_idx = np.argmax(max_per_mooring)
governing_node = mooring_nodes[governing_idx]

df_mooring_summary = pd.DataFrame({
    "mooring_index": np.arange(n_moor),
    "node": mooring_nodes,
    "x [m]": [NodeC[node][0] for node in mooring_nodes],
    "max_Fy [N]": np.max(np.abs(Fy_moor), axis=0),
    "max_Fz [N]": np.max(np.abs(Fz_moor), axis=0),
    "max_Fres [N]": np.max(np.abs(Fres_moor), axis=0),
})

display(df_mooring_summary)
print("Governing mooring node:", governing_node)
print("Maximum resultant mooring load [N]:", max_per_mooring[governing_idx])

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(t, Fy_moor[:, governing_idx], label=r"$F_y$")
plt.plot(t, Fz_moor[:, governing_idx], label=r"$F_z$")
plt.plot(t, Fres_moor[:, governing_idx], label=r"$F_{res}$", linewidth=2)

plt.xlabel("Time [s]")
plt.ylabel("Mooring load [N]")
plt.title(f"Mooring load history at governing node {governing_node}")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(df_mooring_summary["x [m]"], df_mooring_summary["max_Fres [N]"], marker="o")
plt.xlabel("Tunnel coordinate x [m]")
plt.ylabel("Maximum resultant mooring load [N]")
plt.title("Maximum dynamic mooring load envelope")
plt.grid(True)
plt.show()

## Dynamic internal force envelopes

In [ ]:
def beam3d_local_stiffness(EA, EIy, EIz, GJ, L):
    k = np.zeros((12, 12))

    k[0, 0] = k[6, 6] = EA / L
    k[0, 6] = k[6, 0] = -EA / L

    k[3, 3] = k[9, 9] = GJ / L
    k[3, 9] = k[9, 3] = -GJ / L

    c1, c2, c3, c4 = 12*EIz/L**3, 6*EIz/L**2, 4*EIz/L, 2*EIz/L
    dofs = [1, 5, 7, 11]
    kb = np.array([
        [ c1,  c2, -c1,  c2],
        [ c2,  c3, -c2,  c4],
        [-c1, -c2,  c1, -c2],
        [ c2,  c4,  c2,  c3],
    ])
    for i in range(4):
        for j in range(4):
            k[dofs[i], dofs[j]] += kb[i, j]

    c1, c2, c3, c4 = 12*EIy/L**3, 6*EIy/L**2, 4*EIy/L, 2*EIy/L
    dofs = [2, 4, 8, 10]
    kb = np.array([
        [ c1, -c2, -c1, -c2],
        [-c2,  c3,  c2,  c4],
        [-c1,  c2,  c1,  c2],
        [-c2,  c4,  c2,  c3],
    ])
    for i in range(4):
        for j in range(4):
            k[dofs[i], dofs[j]] += kb[i, j]

    return k


def element_dofs(n1, n2):
    return np.r_[
        np.arange(n1*LDOF, n1*LDOF + LDOF),
        np.arange(n2*LDOF, n2*LDOF + LDOF)
    ]


n_elements = len(Ele_tunnel)

N_env  = np.zeros(n_elements)
Vy_env = np.zeros(n_elements)
Vz_env = np.zeros(n_elements)
Tx_env = np.zeros(n_elements)
My_env = np.zeros(n_elements)
Mz_env = np.zeros(n_elements)
x_mid  = np.zeros(n_elements)

for e, ele in enumerate(Ele_tunnel):

    n1 = int(ele[0])
    n2 = int(ele[1])

    L_e = np.linalg.norm(np.asarray(NodeC[n2]) - np.asarray(NodeC[n1]))
    x_mid[e] = 0.5 * (NodeC[n1][0] + NodeC[n2][0])

    k_e = beam3d_local_stiffness(Beam_EA, Beam_EIy, Beam_EIz, Beam_GJ, L_e)
    edofs = element_dofs(n1, n2)

    for n in range(n_steps):

        u_e = U_full[n, edofs]
        f_e = k_e @ u_e

        N_env[e]  = max(N_env[e],  np.max(np.abs([f_e[0], f_e[6]])))
        Vy_env[e] = max(Vy_env[e], np.max(np.abs([f_e[1], f_e[7]])))
        Vz_env[e] = max(Vz_env[e], np.max(np.abs([f_e[2], f_e[8]])))
        Tx_env[e] = max(Tx_env[e], np.max(np.abs([f_e[3], f_e[9]])))
        My_env[e] = max(My_env[e], np.max(np.abs([f_e[4], f_e[10]])))
        Mz_env[e] = max(Mz_env[e], np.max(np.abs([f_e[5], f_e[11]])))


df_force_env = pd.DataFrame({
    "element": np.arange(n_elements),
    "x_mid [m]": x_mid,
    "N_env [N]": N_env,
    "Vy_env [N]": Vy_env,
    "Vz_env [N]": Vz_env,
    "Tx_env [Nm]": Tx_env,
    "My_env [Nm]": My_env,
    "Mz_env [Nm]": Mz_env,
})

display(df_force_env)

display(pd.DataFrame({
    "quantity": ["N", "Vy", "Vz", "Tx", "My", "Mz"],
    "maximum": [
        df_force_env["N_env [N]"].max(),
        df_force_env["Vy_env [N]"].max(),
        df_force_env["Vz_env [N]"].max(),
        df_force_env["Tx_env [Nm]"].max(),
        df_force_env["My_env [Nm]"].max(),
        df_force_env["Mz_env [Nm]"].max(),
    ],
    "unit": ["N", "N", "N", "Nm", "Nm", "Nm"]
}))

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_force_env["x_mid [m]"], df_force_env["My_env [Nm]"], label=r"$M_y$")
plt.plot(df_force_env["x_mid [m]"], df_force_env["Mz_env [Nm]"], label=r"$M_z$")
plt.xlabel("Tunnel coordinate x [m]")
plt.ylabel("Moment envelope [Nm]")
plt.title("Dynamic bending moment envelopes")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(df_force_env["x_mid [m]"], df_force_env["Vy_env [N]"], label=r"$V_y$")
plt.plot(df_force_env["x_mid [m]"], df_force_env["Vz_env [N]"], label=r"$V_z$")
plt.xlabel("Tunnel coordinate x [m]")
plt.ylabel("Shear force envelope [N]")
plt.title("Dynamic shear force envelopes")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(df_force_env["x_mid [m]"], df_force_env["N_env [N]"])
plt.xlabel("Tunnel coordinate x [m]")
plt.ylabel("Axial force envelope [N]")
plt.title("Dynamic axial force envelope")
plt.grid(True)
plt.show()